# **Automated Housing Data Analysis Report - Dundee Housing Sites Dataset**

**Overview**

This report describes an automated analysis of the housingSites.csv dataset, which contains information about housing development sites across Dundee. The analysis was carried out using Python and focuses on site capacity, projected housing builds, and tenure types. The aim was to demonstrate how automation using functions, logging, and a main program structure can make data analysis more reliable and repeatable.

**Data Loading and Preparation**

The dataset was loaded using a **load_data()** function. Columns that were not needed for the analysis were removed using a separate **drop_columns()** function. This helped reduce unnecessary data and allowed the analysis to focus on key housing and planning information. Logging messages were used to confirm that the data was loaded and prepared successfully.

**Data Cleaning**

The data was cleaned using the **clean_data()** function. Text columns such as site addresses and developer names were standardised by removing extra spaces and applying consistent capitalisation. The Tenure type and Site status columns were converted to categorical data types, while numeric columns such as Site capacity and yearly build projections were converted to integers. This ensured that the data was consistent and suitable for calculations.

**Metrics Calculation**

Key housing metrics were calculated using the **calculate_metrics()** function. These included:
- The total site capacity across all housing sites.
- The total number of projected builds between 2024 and 2027.
- The total site capacity grouped by tenure type.

The results showed that private housing has the largest overall site capacity, followed by Registered Social Landlords (RSL), with a smaller amount classified as TBC.

**Filtering and Ranking**

Additional insights were generated using filtering and ranking functions. Sites with a status of Under Construction were filtered to identify active developments. A ranking function was also used to show which tenure types have the highest site capacity.

**Automation and Logging**

All steps of the analysis were organised within a **main()** function, allowing the entire workflow to run automatically from start to finish. Logging was used throughout the program to track progress and confirm when each stage of the analysis was completed.

**Outputs**

The following output files were generated by the program:
- housing_sites_automation_cleaned.csv – the cleaned dataset
- single_metrics.csv – summary statistics
- cum_capacity_by_tenure.csv – site capacity grouped by tenure type

**Conclusion**

This automated approach shows how Python functions, logging, and structured execution can be used to efficiently analyse housing data. The workflow reduces manual effort, improves reliability, and produces clear outputs that can support housing planning and decision-making.


In [14]:
#Import python libraries
import pandas as pd
import logging

In [15]:
#Ignore all warnings
import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings(action='ignore', category=DeprecationWarning)

In [16]:
#Start loggings
logging.basicConfig(level = logging.INFO)

In [17]:
#Define variables
#Columns to drop
columns_to_drop = ['OBJECTID',
                     'Site reference',
                     'LDP2 reference',
                     'Year site added',
                     'Site area (ha)',
                     'Easting',
                     'Northing',
                     'Site type',
                     'Planning application reference',
                     'Last planning approval date',
                     'Building warrant reference',
                     'Date completed/expired',
                     'Greenfield/Brownfield',
                     'Self build',
                     'Windfall site',
                     'BW plots written off',
                     'No of houses',
                     'No of flats',
                     'Plots completed in survey year',
                     'Total completions',
                     'Units to build',
                     'Year 28/29',
                     'Year 29/30',
                     'Year 30/31',
                     'Year 31/32',
                     'Year 32/33',
                     'Year 33/34',
                     'Later Years',
                     'Total Programmed',
                     'Shape__Area',
                     'Shape__Length']

In [18]:
#Define functions

#Function to load dataset
def load_data(csv_name : str = 'housingSites'):
  '''
  Load dataset.

  Parameters:
    cvs_name(str) : Name of the csv file to load.

  Returns:
    pd.DataFrame : DataFrame of the csv data.
  '''
  return pd.read_csv(f'{csv_name}.csv')

#Function to drop columns with not-relevant information to asnwer the quetions
def drop_columns(data_df : pd.DataFrame, columns_to_drop : list[str]):
  '''
  Drop columns with not-relevant information to answer the questions.

  Parameters:
    data_df (pd.DataFrame): The DataFrame to drop columns from.
    columns_to_drop (str): Names of columns to drop.

  Returns:
    pd.DataFrame: DataFrame with dropped columns.
  '''

  #To work with the copy of the data
  data_df = data_df.copy()

  #Drop columns
  data_df.drop(columns = columns_to_drop, inplace = True)

  #Return data
  logging.info('Data loaded')
  return data_df

#Function to clean dataset
def clean_data(data_df : pd.DataFrame):
  '''
  Clean the dataset.

  Parameters:
   data_df(pd.DataFrame) : The DataFrame to clean.

  Returns:
    pd.DataFrame : Cleaned DataFrame
  '''

  #To work with the copy of the dataset
  data_df = data_df.copy()

  #Convert object datatype: Tenure type and Site status to category and all other columns to string
  #Convert all other columns to int64 datatype
  for column in data_df.columns:
    if column == 'Tenure type' or column == 'Site status':
      data_df[column] = data_df[column].astype('category')
    elif data_df[column].dtype == 'object':
      data_df[column] = data_df[column].astype('string')
    else:
      data_df[column] = data_df[column].astype('int64')

  #Standardize capitalization, and removing extra spaces
  for column in data_df.columns:
    if data_df[column].dtype == 'string':
      data_df[column] = data_df[column].str.strip().str.title()

  #Return data
  logging.info('Data cleaned')
  return data_df


#Functions for calculation
def calculate_metrics(data_df : pd.DataFrame):
  '''
  Calculate metrics for the dataset - total site capacity,
                                      projects builds for 2024-2027,
                                      cumulative capacity per tenure type stored in dictionaries.

  Parameters:
    data_df(pd.DataFrame) : The DataFrame used for calculation.

  Returns:
    total_site_capacity(int64) : Total site capacity.
    projects_builds_2024_2027(int64) : Projects builds for 2024-2027.
    cumulative_capacity_by_tenure(dict) : Cumulative capacity by tenure type.
  '''

  #To work with the copy of the data
  data_df = data_df.copy()

  #Total site capacity
  total_site_capacity = data_df['Site capacity'].sum()

  #Projects builds for 2024-2027
  year_columns = ['Year 24/25', 'Year 25/26', 'Year 26/27', 'Year 27/28']

  projects_builds_2024_2027 = data_df[year_columns] \
                                        .sum() \
                                        .sum()

  #Cumulative capacity by tenure type
  cumulative_capacity_by_tenure = data_df \
                                  .groupby('Tenure type')['Site capacity'] \
                                  .sum() \
                                  .sort_values(ascending = False) \
                                  .to_dict()

  #Create DataFrame wih the calcualted single metrics
  single_metrics = {'Total site capacity' : total_site_capacity,
                    'Projects build for 2024-2027' : projects_builds_2024_2027}

  #single_metrics_df = pd.DataFrame.from_dict(single_metrics)

  single_metrics_df = pd.DataFrame()
  single_metrics_df['Single Metric'] = single_metrics.keys()
  single_metrics_df['Single Metric Value'] = single_metrics.values()

  #Create DataFrame with the calcualted Cumulative capacity by tenure
  cumulative_capacity_df = pd.DataFrame()
  cumulative_capacity_df['Tenure type'] = cumulative_capacity_by_tenure.keys()
  cumulative_capacity_df['Site capacity'] = cumulative_capacity_by_tenure.values()



  #cum_capacity_by_tenure_df = pd.DataFrame.from_dict(cumulative_capacity_by_tenure)


  #Returns
  logging.info('Metrics calculated')
  return single_metrics_df, cumulative_capacity_df

#Save data
def save_data(data_df : pd.DataFrame, filepath : str):
  '''
  Save dataset.

  Parameters:
    data_df(pd.DataFrame) : DataFrame to save.
    filepath(str) : Path to save the file.

  Returns:
    None
  '''

  data_df.to_csv(filepath, index = False)
  logging.info('Data saved')


#Function - top_sites_by_capacity
def top_sites_by_capacity(df :pd.DataFrame, n : int = 3):
  '''
  Print the top n sites by capacity.

  Parameters:
    df(pd.DataFrame) : DataFrame to print.
    n(int) : Number of the top sites to print.

  Returns:
    None
  '''
  print(df.head(n))
  logging.info('Top sites printed')

#Function filter_sites
def filter_sites(df : pd.DataFrame, site_status : str = None):
  '''
  Filter the dataset by site status.

  Parameters:
    df(pd.DataFrame) : DataFrame to filter.
    site_status(str) : Site status to filter.

  Returns:
    pd.DataFrame : Filtered DataFrame.
  '''
  if site_status is None:
    logging.info('No filter applied')
    return df
  else:
    logging.info(f'Data is filtered for {site_status}')
    return df[df['Site status'] == site_status]





In [19]:
#Main
def main():
  data_df = load_data()
  data_df = drop_columns(data_df, columns_to_drop)
  data_df = clean_data(data_df)
  save_data(data_df, 'housing_sites_automation_cleaned.csv')
  df_1, df_2 = calculate_metrics(data_df)
  top_sites_by_capacity(df_2)
  filtered_data = filter_sites(data_df, 'Under Construction')
  print(filtered_data)
  save_data(df_1, 'single_metrics.csv')
  save_data(df_2, 'cum_capacity_by_tenure.csv')




In [20]:
#Run program
main()


  Tenure type  Site capacity
0     Private           3503
1         RSL            840
2         TBC            419
                                          Site address         Site status  \
1                Riverside Drive, Former Homebase Site  Under Construction   
2                           Monifieth Road, Armitstead  Under Construction   
4                East School Road, Former Downfield Ps  Under Construction   
5                St Leonard Place, Former Macalpine Ps  Under Construction   
7                                  Seagate/Trades Lane  Under Construction   
..                                                 ...                 ...   
151                      Central Waterfront - Site 6 *  Under Construction   
152  Summerfield Avenue At Summerfield Gardens, Lan...  Under Construction   
153  Summerfield Avenue At Summerfield Gardens, Lan...  Under Construction   
169                  Strathern Road, 32, Garden Ground  Under Construction   
177                     Bu